In [ ]:
# ==========================================
# Cell 1: 基础配置与 CNN-BiLSTM 架构 (V4.1.0)
# ==========================================
import os, pathlib, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR # 新增：余弦退火学习率调度器
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('/jovyan/QMJ10')  # 你的根目录
DATA_DIR = ROOT / 'data' / 'PEMS-BAY'
OUT_DIR = ROOT / 'output_410_transfer_advanced' # 开辟新的输出目录防止覆盖
CM_DIR = OUT_DIR / 'confusion_matrices'
MODEL_DIR = OUT_DIR / 'models'

for d in [CM_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

# 保持你原来的 V3.0 完美适配的 144 长度架构不变
class CNN_BiLSTM_Model(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout1d(0.1),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout1d(0.1)
        )
        self.lstm = nn.LSTM(
            input_size=64, hidden_size=64, 
            num_layers=2, batch_first=True, bidirectional=True, dropout=0.2
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(128 + 3, 64), nn.ReLU(), nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        cnn_out = self.cnn(x) 
        lstm_in = cnn_out.permute(0, 2, 1) 
        lstm_out, _ = self.lstm(lstm_in) 
        lstm_out_pool = lstm_out.permute(0, 2, 1) 
        features = self.gap(lstm_out_pool).view(lstm_out_pool.size(0), -1) 
        return self.fc(torch.cat([features, stats], dim=1))

In [ ]:
# ==========================================
# Cell 2: PEMS-BAY 解析、降采样与 Dataset (带增强)
# ==========================================
def apply_kalman_filter(curve, process_variance=1e-4, measurement_variance=1e-2):
    n = len(curve)
    if n == 0: return curve
    xhat, P = np.zeros(n), np.zeros(n)
    xhat[0], P[0] = curve[0], 1.0
    for k in range(1, n):
        xhat_minus = xhat[k-1]
        P_minus = P[k-1] + process_variance
        K = P_minus / (P_minus + measurement_variance) 
        xhat[k] = xhat_minus + K * (curve[k] - xhat_minus)
        P[k] = (1 - K) * P_minus
    return xhat

def _process_curve(curve: np.ndarray):
    curve = np.nan_to_num(curve, nan=np.nanmean(curve) if not np.isnan(curve).all() else 0.0)
    curve = apply_kalman_filter(curve)

    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom 
    
    mean_val, std_val = np.mean(norm_curve), np.std(norm_curve)
    log_max_val = np.log1p(max_val) / 10.0 
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    grad_curve = np.gradient(norm_curve)
    return np.stack([norm_curve, grad_curve], axis=0).astype(np.float32), stats

class PEMS_BAY_Dataset(Dataset):
    def __init__(self, h5_path, target_days=None, is_train=False):
        self.is_train = is_train # 新增：标识是否为训练集
        print(f">>> 正在加载 PEMS-BAY 数据库: {h5_path} (Training Mode: {is_train})")
        df = pd.read_hdf(h5_path)
        df.index = pd.to_datetime(df.index)
        
        days_group = df.groupby(df.index.date)
        self.samples = []
        valid_day_count = 0
        
        for date, group in days_group:
            if len(group) != 288: continue
            if target_days is not None and valid_day_count not in target_days:
                valid_day_count += 1
                continue
                
            y = date.weekday() 
            downsampled_data = group.iloc[::2].values 
            
            for sid in range(downsampled_data.shape[1]):
                curve_144 = downsampled_data[:, sid]
                seq, stats = _process_curve(curve_144)
                meta = {"station_id": sid, "date": str(date)}
                self.samples.append((seq, stats, y, meta))
                
            valid_day_count += 1
            
        self.n = len(self.samples)
        print(f"提取完成！包含 {valid_day_count if target_days is None else len(target_days)} 天的数据，共计 {self.n} 条时序样本。")

    def __len__(self): return self.n
    
    def __getitem__(self, idx):
        seq, stats, y, _ = self.samples[idx]
        seq_tensor = torch.from_numpy(seq).clone()
        
        # 【核心升级 2】：仅在训练阶段加入动态时序增强，对抗极少样本过拟合
        if self.is_train:
            # 随机幅度缩放 (模拟流量的轻微波动，±5%)
            scale = 1.0 + (torch.rand(1).item() - 0.5) * 0.1
            seq_tensor[0, :] *= scale 
            
            # 注入微小高斯噪声 (模拟传感器采集误差)
            noise = torch.randn_like(seq_tensor[0, :]) * 0.01
            seq_tensor[0, :] += noise

        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long)

h5_file = DATA_DIR / 'pems-bay.h5'
few_shot_days = set(range(0, 14)) 
test_days = set(range(14, 180))

# 实例化时区分训练和测试
ds_transfer_train = PEMS_BAY_Dataset(h5_file, target_days=few_shot_days, is_train=True)
ds_transfer_test  = PEMS_BAY_Dataset(h5_file, target_days=test_days, is_train=False)

loader_train = DataLoader(ds_transfer_train, batch_size=256, shuffle=True)
loader_test  = DataLoader(ds_transfer_test, batch_size=512, shuffle=False)

In [ ]:
# ==========================================
# Cell 3: 领域自适应 (Advanced Transfer Learning)
# ==========================================
model = CNN_BiLSTM_Model().to(DEVICE)
# 请确保这里的路径是你源域模型的正确路径
pretrained_path = ROOT / 'output_300' / 'models' / 'cnn_lstm_v300_global.pth' 

if pretrained_path.exists():
    model.load_state_dict(torch.load(pretrained_path))
    print(f"成功加载预训练老模型 (源域): {pretrained_path}")
else:
    print(f"警告: 找不到预训练模型 {pretrained_path}！将使用随机初始化权重训练。")

# 【核心升级 1】：解冻所有层，准备进行差分学习率微调
print(">>> 正在准备差分微调，解冻全部网络层...")
for param in model.parameters():
    param.requires_grad = True

active_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"参与微调的参数量 (全局): {active_params}")

EPOCHS = 40 # 可以稍微增加一点 Epoch，因为有 Scheduler 和 Data Augmentation

# 【核心升级 1 续】：差分学习率设置
optimizer = optim.AdamW([
    {'params': model.cnn.parameters(), 'lr': 1e-5},   # 底层特征：极小学习率，保留源域共性
    {'params': model.lstm.parameters(), 'lr': 1e-4},  # 时序逻辑：较小学习率，适应新波形
    {'params': model.fc.parameters(), 'lr': 2e-3}     # 分类头：正常学习率，快速收敛
], weight_decay=1e-3)

# 【核心升级 3】：引入余弦退火学习率调度器
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_transfer_model(model, loader_train, loader_test, epochs=40):
    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        
        # 微调训练
        for x, stats, y in loader_train:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, stats)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # 更新学习率
        scheduler.step()
            
        # 测试集评估
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, stats, y in loader_test:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                preds = torch.argmax(model(x, stats), dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
                
        test_acc = correct / total if total > 0 else 0
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), MODEL_DIR / 'cnn_lstm_v410_transfer.pth')
            
        if epoch % 5 == 0 or epoch == 1:
            current_lr_fc = optimizer.param_groups[2]['lr'] # 打印 FC 层的当前 LR 看看衰减情况
            print(f"[Epoch {epoch}/{epochs}] Loss: {total_loss/len(loader_train):.4f} | Target Acc: {test_acc:.2%} | FC LR: {current_lr_fc:.6f}")
            
    return best_acc

print("\n>>> 开始进行目标域 (PEMS-BAY) 高级微调训练...")
final_transfer_acc = train_transfer_model(model, loader_train, loader_test, epochs=EPOCHS)
print(f"\n🎉 高级迁移学习完成！最终泛化准确率达到: {final_transfer_acc:.2%}")

In [ ]:
# ==========================================
# Cell 4: 评估最终结果并绘制混淆矩阵
# ==========================================
model.load_state_dict(torch.load(MODEL_DIR / 'cnn_lstm_v410_transfer.pth'))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, stats, y in loader_test:
        x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
        preds = torch.argmax(model(x, stats), dim=1)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)

plt.figure(figsize=(10, 8))
labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='magma', xticklabels=labels, yticklabels=labels)
plt.title(f"Target Domain (PEMS-BAY) Adv. Transfer Learning", fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()

save_path = CM_DIR / 'matrix_transfer_pemsbay_v410.png'
plt.savefig(save_path, dpi=300)
print(f"目标域混淆矩阵已保存: {save_path}")